<span style="font-size:4em;">Multi Session Analysis </span>



Gather all sessions with a defined samesite ID and align them using CellReg.


Author: Leon Kremers, 2026

# Init Code

In [ ]:
# modules are reloaded before code execution 
# allows to make changes to external functions and test them without having to restart the kernel
%load_ext autoreload
%autoreload 2

In [ ]:
# limit the amount of RAM that this notebook can use
import resource
# Set memory limit (e.g., 256GB in bytes)
memory_limit = 256 * 1024 * 1024 * 1024
resource.setrlimit(resource.RLIMIT_AS, (memory_limit, memory_limit))

In [ ]:
import os

# change to the upper level folder to detect dj_local_conf.json
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
assert os.path.basename(os.getcwd()) == "adamacs", "Please move to the main directory"

from adamacs.pipeline import (
    subject,
    session,
    equipment,
    surgery,
    event,
    trial,
    imaging,
    behavior,
    scan,
    model,
)
import datajoint as dj
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import subprocess
import cv2
import glob

from helpers.pipeline_blocks import (
    define_scans,
    import_CellReg_output,
    get_indices,
    get_all_mask_ids,
)

dj.__version__

# Mastercells

Define scans and loop through analysis functions for each scan.

In [ ]:
# Define Scans
# Set same site ID
samesite_id = "sess9FQ644X9"
# Define stimulus type
stim_type = "Natural Images"  # leave empty for all same_site sessions
# Define which S2P paramset was sused
paramset = 10

scans, curation_keys, samesite_scan_key = define_scans(samesite_id, stim_type, paramset)

# CellReg Prep Code

Prepare CellReg analysis. Find Fall.mat files and copy them to folder. Save folder path in .mat file 

In [ ]:
# Create output folder
import os
import shutil
import scipy.io

site_session_key = (session.Session & f'session_id = "{samesite_id}"').fetch("KEY")
site_scan_key = (scan.Scan & site_session_key).fetch("KEY")[0]
path = "/datajoint-data/data/leonk/analysis/cellreg/"
mouse_id = (session.Session & f'session_id = "{samesite_id}"').fetch("subject")[0]
output_path = path + "LE_" + mouse_id + "_" + samesite_id + "_" + stim_type + "_CR/"
if os.path.exists(output_path):
    print("Output folder already exists")
else:
    os.mkdir(output_path)
print("Output path:" + output_path)

In [ ]:
# decide whether to run CellReg or not
# if CellReg output already exists, set to False
run_cellreg = False

In [ ]:
if run_cellreg:
    # Find Suite2P output
    for i in range(len(scans)):

        print("========================")
        print("Now copying:" + scans[i])

        # Find Suite2P files and load Fall.mat
        scansi = scans[i]
        scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch("KEY")[0]
        data_path = (scan.ScanPath & f'scan_id = "{scansi}"').fetch("path")[0]
        data_path = data_path + "/suite2p/plane0/Fall.mat"
        print(data_path)
        print("========================")
        print("")

        # Copy Fall.mat file into folder
        output = output_path + "/Fall_" + str(i + 1).zfill(2) + "_" + scansi + ".mat"
        shutil.copy(data_path, output)

    # Save output folder to .mat file so it can be imported into CellReg
    data = {"folder_path": output_path}
    scipy.io.savemat("/home/leonk/adamacs/notebooks/temp/temp_folder.mat", data)

Define CellReg settings and save them to .mat file in output_path

In [ ]:
if run_cellreg:
    # CellReg settings
    settings = {
        "input_format": "Suite2p",
        "memory_efficient_run": False,  # either true or false
        "microns_per_pixel": 1.2,
        "alignment_type": "Translations and Rotations",  # either 'Translations', 'Translations and Rotations' or 'Non-rigid'
        "use_parallel_processing": True,  # either true or false
        "maximal_rotation": 10.0,  # in degrees - only relevant if 'Translations and Rotations' is used
        "transformation_smoothness": 2.0,  # levels of non-rigid FOV transformation smoothness (range 0.5-3)
        "reference_session_index": 5,
        "sufficient_correlation_centroids": 0.1,  # For alignment_type = Translations and Rotations: smaller correlation imply no similarity between sessions
        "sufficient_correlation_footprints": 0.1,  # For alignment_type = Translations and Rotations: smaller correlation imply no similarity between sessions
        "maximal_distance": 12.0,  # cell-pairs that are more than __ micrometers apart are assumed to be different cells (10-12 recommended for 2P data)
        "p_same_certainty_threshold": 0.75,  # certain cells are those with p_same>threshld or <1-threshold
        "initial_registration_type": "best_model_string",  # either 'Spatial correlation', 'Centroid distance', or 'best_model_string';
        "registration_approach": "Probabilistic",  # either 'Probabilistic' or 'Simple threshold'
        "model_type": "best_model_string",  # either 'Spatial correlation' or 'Centroid distance'
        "p_same_threshold": 0.3,  # only relevant if probabilistic approach is used
    }

    # Save settings to .mat file
    scipy.io.savemat((output_path + "settings.mat"), settings)
    print("Settings saved!")

Now run CellReg script with Matlab!

In [ ]:
if run_cellreg:

    # define the memory limit for MATLAB
    memory_limit = 256 * 1024 * 1024 * 1024  # 256 GB in bytes

    # Define the MATLAB script you want to run
    matlab_script = "/home/leonk/adamacs/helpers/Leon_CellReg.m"

    # Call MATLAB from Python
    subprocess.run(
        ["bash", "-c", f"ulimit -v {memory_limit // 1024} && /home/leonk/Matlab/bin/matlab -batch \"run('{matlab_script}')\""],
        check=True,
    )

# Analyze Multisession-Aligned Neurons

Import CellReg output

In [ ]:
# Import CellReg output
cell_to_index_map = import_CellReg_output(samesite_id, stim_type)

# Ingest CellReg Output into Datajoint

In [ ]:
# convert cell_to_index_map to python numbering
cell_to_index_map_python = cell_to_index_map - 1

# CellReg ID is ingested into imaging.Segementation.Mask as roi_group
# loop though all scans
for i in range(len(scans)):
    def_scan = scans[i]
    curation_key = curation_keys[i]
    masks = (imaging.Segmentation.Mask & curation_key).fetch("mask")

    # loop through all masks and update roi_group
    for j in range(len(masks)):
        mask = masks[j]
        # get mask key
        mask_key = (imaging.Segmentation.Mask & curation_key & f'mask = "{j}"').fetch(
            "KEY"
        )[0]
        # extract roi_group from cell_to_index_map
        try:
            new_roi_group = np.where(cell_to_index_map_python[:, i] == j)[0][0]
        except:
            new_roi_group = -1
        update_data = {"roi_group": new_roi_group}
        # Update roi_group in DJ
        imaging.Segmentation.Mask.update1({**mask_key, **update_data})

Find units present across all sessions using DJ

In [ ]:
# scans = scans[0:11]
# curation_keys = [curation_keys[i] for i in range(11)]
# samesite_scan_key = samesite_scan_key[0:11]
# print(scans)

In [ ]:
indices, neuron_only_indices = get_indices(samesite_id, scans, curation_keys, stim_type)

print("Possible indices are:")
print(indices)
print(len(indices))
print("Neuron only indices are:")
print(neuron_only_indices)
print(len(neuron_only_indices))

mask_ids_all = get_all_mask_ids(samesite_id, scans, curation_keys, stim_type)

Test whether ingest succesfull and same as CellReg output

In [ ]:
neuron_only_indices[0]

In [ ]:
# test whether ingest was successfull
unit_idx = neuron_only_indices[0]
# extract neuron IDs for unit from each session and convert to python indexing format from DJ
mask_ids_DJ = mask_ids_all[unit_idx]

# extract neuron IDs for unit from each session and convert to python indexing format from cell_to_index_map
mask_ids_CR = cell_to_index_map[unit_idx]
mask_ids_CR = [int(x - 1) for x in mask_ids_CR]
print("CellReg IDs:")
print(mask_ids_CR)
print("DataJoint IDs:")
print(list(mask_ids_DJ))


# plot average images
scalemin = 0
scalemax = 99
ncols = len(scans)
nrows = 2
fig, axes = plt.subplots(ncols=ncols, nrows=nrows, figsize=(ncols * 4.5, nrows * 5))
for i in range(nrows):
    for j in range(ncols):
        ax = axes[i, j]
        try:
            scansi = scans[j]
            scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch("KEY")[0]
            latest_curation = (imaging.Curation & scan_key).fetch("curation_id").max()
            curation_key = (
                imaging.Curation
                & scan_key
                & f"curation_id={latest_curation}"
                & f"paramset_idx={paramset}"
            ).fetch("KEY")[0]
            average_image = (
                imaging.MotionCorrection.Summary & curation_key & "field_idx=0"
            ).fetch1("average_image")
            cmin = np.percentile(average_image, scalemin)
            cmax = np.percentile(average_image, scalemax)
            if i == 0:
                if j == 0:
                    ax.set_title("CellReg IDs", fontsize=20)
                neuron = mask_ids_CR[j]
            else:
                if j == 0:
                    ax.set_title("DataJoint IDs", fontsize=20)
                neuron = mask_ids_DJ[j]
            # fetch center coordinates from datajoint
            ymiddle = int(
                (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch(
                    "mask_center_y"
                )
            )
            xmiddle = int(
                (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch(
                    "mask_center_x"
                )
            )
            # patch size is 50 pixel
            ycorner1 = ymiddle - 25
            xcorner1 = xmiddle - 25
            ycorner2 = ymiddle + 25
            xcorner2 = xmiddle + 25
            # extract patch from average image
            patch = average_image[ycorner1:ycorner2, xcorner1:xcorner2]
            # plot patch
            ax.imshow(patch, cmap="gray", vmin=cmin, vmax=cmax)
            ax.axis("off")

        except:
            ax.remove()

# Adjust layout to prevent overlap
plt.tight_layout()
# Show the plot
plt.show()

# Extract images for all sessions

Display whole FOV mean image for all sessions

In [ ]:
# Display mean images
scalemin = 0
scalemax = 99
ncols = len(scans)
nrows = int(np.ceil(len(scans) / ncols))

fig, axes = plt.subplots(ncols=ncols, figsize=(ncols * 4.5, nrows * 4.5))


for y in range(ncols):
    i = y
    ax = axes[y]
    try:
        scansi = scans[i]
        scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch("KEY")[0]
        average_image = (
            imaging.MotionCorrection.Summary & scan_key & "field_idx=0"
        ).fetch("average_image")[0]
        cmin = np.percentile(average_image, scalemin)
        cmax = np.percentile(average_image, scalemax)
        ax.imshow(average_image, cmap="gray", vmin=cmin, vmax=cmax)
        ax.set_title(scansi)
        ax.axis("off")
        scalebar = AnchoredSizeBar(
            ax.transData,
            # 100 µm is the size of the scalebar
            83,
            "",
            "lower right",
            pad=1,
            color="white",
            frameon=False,
            label_top=True,
            size_vertical=5,
        )
        ax.add_artist(scalebar)
    except:
        ax.remove()
# Adjust layout to prevent overlap
plt.tight_layout()
# Show the plot
plt.show()

Plot cell number across sessions

In [ ]:
# get neuron number for each image session
nneurons = []
for i in range(len(scans)):
    curation_key = curation_keys[i]
    soma_masks = (
        imaging.Segmentation.Mask * imaging.MaskClassification.MaskType
        & curation_keys[i]
        & "mask_type='soma'"
    ).fetch("mask")
    nneurons.append(len(soma_masks))

# plot neuron number across sessions
plt.plot(nneurons, label="Neurons")
plt.xlabel("Session")
plt.ylabel("Number of neurons")
plt.ylim(0, max(nneurons) + 50)
plt.show()

Display CellReg output figures

In [ ]:
# load output figures
mouse_id = (session.Session & f'session_id = "{samesite_id}"').fetch("subject")[0]
results_path = (
    "/datajoint-data/data/leonk/analysis/cellreg/LE_"
    + mouse_id
    + "_"
    + samesite_id
    + "_"
    + stim_type
    + "_CR/Results/"
)
figures_path = results_path + "Figures/"
results_images = sorted(glob.glob(os.path.join(figures_path, "*.png")))
file_names = sorted([os.path.basename(file) for file in results_images])


# Display the figures
ncols = 5
nrows = int(np.ceil(len(file_names) / ncols))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 4.5, nrows * 4.5))

for x in range(nrows):
    for y in range(ncols):
        i = (x * ncols) + y
        ax = axes[x, y]
        try:
            image = os.path.join(results_images[i])
            im = cv2.imread(image)
            ax.imshow(im)
            ax.set_title(file_names[i][:-4])
            ax.axis("off")
        except:
            ax.remove()
# Adjust layout to prevent overlap
plt.tight_layout()
# Show the plot
plt.show()

In [ ]:
# Only display final registration figure
final_registration = figures_path + file_names[-1]
im = cv2.imread(final_registration)
plt.figure(figsize=(10, 10))
plt.imshow(im)
plt.title(file_names[-1][:-4])
plt.axis("off")

Display selected unit in all sessions

In [ ]:
# Find indices that are present across all sessions using DJ
indices = np.where((cell_to_index_map != 0).all(axis=1))[0]

print("Possible indices are:")
print(indices)

In [ ]:
len(indices)

In [ ]:
# define unit index
unit_idx = indices[0]
print(unit_idx)

In [ ]:
# extract neuron IDs for unit from each session and convert to python indexing format
neuron_ids = cell_to_index_map[unit_idx]
neuron_ids = [int(x - 1) for x in neuron_ids]
print(neuron_ids)

# check whether unit was present for all sessions
temp = False
for i in range(len(neuron_ids)):
    if neuron_ids[i] < 0:
        temp = True
# Give error when unit not present across all sessions
if temp == True:
    raise ValueError("Unit not present across all sessions")
# Continue analysis if unit present across all session
else:
    ncols = 11
    nrows = int(np.ceil(len(neuron_ids) / ncols))
    fig, axes = plt.subplots(ncols=ncols, figsize=(ncols * 4.5, nrows * 4.5))

    for y in range(ncols):
        i = y
        ax = axes[y]
        try:
            scansi = scans[i]
            scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch("KEY")[0]
            latest_curation = (imaging.Curation & scan_key).fetch("curation_id").max()
            curation_key = (
                imaging.Curation
                & scan_key
                & f"curation_id={latest_curation}"
                & f"paramset_idx={paramset}"
            ).fetch("KEY")[0]
            average_image = (
                imaging.MotionCorrection.Summary & curation_key & "field_idx=0"
            ).fetch1("average_image")
            cmin = np.percentile(average_image, scalemin)
            cmax = np.percentile(average_image, scalemax)
            neuron = int(neuron_ids[i])
            # fetch center coordinates from datajoint
            ymiddle = int(
                (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch(
                    "mask_center_y"
                )
            )
            xmiddle = int(
                (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch(
                    "mask_center_x"
                )
            )
            # patch size is 50 pixel
            ycorner1 = ymiddle - 25
            xcorner1 = xmiddle - 25
            ycorner2 = ymiddle + 25
            xcorner2 = xmiddle + 25
            # extract patch from average image
            patch = average_image[ycorner1:ycorner2, xcorner1:xcorner2]
            # plot patch
            ax.imshow(patch, cmap="gray", vmin=cmin, vmax=cmax)
            ax.axis("off")
            # add  scalebar
            scalebar = AnchoredSizeBar(
                ax.transData,
                # 5 µm is the size of the scalebar
                8.3,
                "",
                "lower right",
                pad=1,
                color="white",
                frameon=False,
                label_top=True,
                size_vertical=1.5,
            )
            ax.add_artist(scalebar)

        except:
            ax.remove()
# Adjust layout to prevent overlap
plt.tight_layout()
# Show the plot
plt.show()

In [ ]:
from scipy.ndimage import binary_dilation

patch_size = 50
half = patch_size // 2

ncols = len(scans)
nrows = len(neuron_only_indices)
fig, axes = plt.subplots(ncols=ncols, nrows=nrows, figsize=(ncols * 2, nrows * 2))

for x in range(nrows):
    unit_idx = neuron_only_indices[x]
    mask_id_DJ = mask_ids_all[unit_idx]

    for y in range(ncols):
        i = y
        ax = axes[x, y]
        try:
            scansi = scans[i]
            scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch("KEY")[0]
            latest_curation = (
                (imaging.Curation & scan_key).fetch("curation_id").max()
            )
            curation_key = (
                imaging.Curation
                & scan_key
                & f"curation_id={latest_curation}"
                & f"paramset_idx={paramset}"
            ).fetch("KEY")[0]
            average_image = (
                imaging.MotionCorrection.Summary & curation_key & "field_idx=0"
            ).fetch1("average_image")
            cmin = np.percentile(average_image, scalemin)
            cmax = np.percentile(average_image, scalemax)
            neuron = int(mask_id_DJ[i])
            # fetch center coordinates + pixel mask from datajoint
            ymiddle = int(
                (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch(
                    "mask_center_y"
                )
            )
            mask_ypix = (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch1(
                    "mask_ypix"
                )
            xmiddle = int(
                (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch(
                    "mask_center_x"
                )
            )
            mask_xpix = (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch1(
                    "mask_xpix"
                )

            # Pad the image so the patch is always 50x50, even at borders
            img_h, img_w = average_image.shape
            padded_image = np.pad(
                average_image, pad_width=half, mode="constant",
                constant_values=np.min(average_image)
            )
            # Shift center into padded coordinates
            yc = ymiddle + half
            xc = xmiddle + half
            # Extract guaranteed 50x50 patch
            patch = padded_image[yc - half : yc + half, xc - half : xc + half]

            # Plot patch
            ax.imshow(patch, cmap="gray", vmin=cmin, vmax=cmax)

            # Build binary mask on padded image, then extract same patch region
            mask_img = np.zeros_like(padded_image, dtype=bool)
            my = mask_ypix + half
            mx = mask_xpix + half
            valid = (my >= 0) & (my < padded_image.shape[0]) & (mx >= 0) & (mx < padded_image.shape[1])
            mask_img[my[valid], mx[valid]] = True
            mask_patch = mask_img[yc - half : yc + half, xc - half : xc + half]

            # Compute contour: dilated mask minus original = border pixels only
            border = binary_dilation(mask_patch, iterations=2) & ~mask_patch
            border_overlay = np.zeros((*mask_patch.shape, 4))  # RGBA
            border_overlay[border] = [1, 0, 0, 0.8]  # red border
            ax.imshow(border_overlay)
            #also plot inside of mask but make it transparent
            inside_overlay = np.zeros((*mask_patch.shape, 4))  # RGBA
            inside_overlay[mask_patch] = [1, 0, 0, 0.3]  # transparent red inside
            ax.imshow(inside_overlay)

            if y == 0:
                ax.set_ylabel(unit_idx, fontsize=20, rotation=0, pad=20)
                ax.set_yticks([])
                ax.set_xticks([])
            else:
                ax.axis("off")
            scalebar = AnchoredSizeBar(
                ax.transData,
                # 5 µm is the size of the scalebar
                8.3,
                "",
                "lower right",
                pad=0.2,
                color="white",
                frameon=False,
                label_top=True,
                size_vertical=1.5,
            )
            ax.add_artist(scalebar)

        except:
            ax.remove()
# Adjust layout to prevent overlap
plt.tight_layout()
# Show the plot
plt.show()

In [ ]:
from scipy.ndimage import binary_dilation

patch_size = 50
half = patch_size // 2

# Number of neuron indices per column group
group_size = 60
n_sessions = len(scans)
n_neurons = len(neuron_only_indices)
n_groups = int(np.ceil(n_neurons / group_size))
nrows = min(n_neurons, group_size)

# Each group gets n_sessions columns, plus 1 spacer column between groups
spacer_cols = n_groups - 1
total_cols = n_groups * n_sessions + spacer_cols

fig, axes = plt.subplots(
    ncols=total_cols, nrows=nrows,
    figsize=(total_cols * 2, nrows * 2),
    gridspec_kw={"wspace": 0.05, "hspace": 0.05},
)

for g in range(n_groups):
    start_idx = g * group_size
    end_idx = min(start_idx + group_size, n_neurons)
    col_offset = g * (n_sessions + 1)  # +1 for spacer column

    for x in range(nrows):
        neuron_row = start_idx + x

        # Handle spacer column: hide it
        if g < n_groups - 1:
            spacer_col = col_offset + n_sessions
            axes[x, spacer_col].axis("off")

        for y in range(n_sessions):
            ax = axes[x, col_offset + y]

            # If this row exceeds the neurons in this group, hide the axis
            if neuron_row >= end_idx:
                ax.axis("off")
                continue

            unit_idx = neuron_only_indices[neuron_row]
            mask_id_DJ = mask_ids_all[unit_idx]

            try:
                scansi = scans[y]
                scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch("KEY")[0]
                latest_curation = (
                    (imaging.Curation & scan_key).fetch("curation_id").max()
                )
                curation_key = (
                    imaging.Curation
                    & scan_key
                    & f"curation_id={latest_curation}"
                    & f"paramset_idx={paramset}"
                ).fetch("KEY")[0]
                average_image = (
                    imaging.MotionCorrection.Summary & curation_key & "field_idx=0"
                ).fetch1("average_image")
                cmin = np.percentile(average_image, scalemin)
                cmax = np.percentile(average_image, scalemax)
                neuron = int(mask_id_DJ[y])
                # fetch center coordinates + pixel mask from datajoint
                ymiddle = int(
                    (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch(
                        "mask_center_y"
                    )
                )
                mask_ypix = (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch1(
                        "mask_ypix"
                    )
                xmiddle = int(
                    (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch(
                        "mask_center_x"
                    )
                )
                mask_xpix = (imaging.Segmentation.Mask & curation_key & f"mask={neuron}").fetch1(
                        "mask_xpix"
                    )

                # Pad the image so the patch is always 50x50, even at borders
                img_h, img_w = average_image.shape
                padded_image = np.pad(
                    average_image, pad_width=half, mode="constant",
                    constant_values=np.min(average_image)
                )
                # Shift center into padded coordinates
                yc = ymiddle + half
                xc = xmiddle + half
                # Extract guaranteed 50x50 patch
                patch = padded_image[yc - half : yc + half, xc - half : xc + half]

                # Plot patch
                ax.imshow(patch, cmap="gray", vmin=cmin, vmax=cmax)

                # Build binary mask on padded image, then extract same patch region
                mask_img = np.zeros_like(padded_image, dtype=bool)
                my = mask_ypix + half
                mx = mask_xpix + half
                valid = (my >= 0) & (my < padded_image.shape[0]) & (mx >= 0) & (mx < padded_image.shape[1])
                mask_img[my[valid], mx[valid]] = True
                mask_patch = mask_img[yc - half : yc + half, xc - half : xc + half]

                # Compute contour: dilated mask minus original = border pixels only
                border = binary_dilation(mask_patch, iterations=2) & ~mask_patch
                border_overlay = np.zeros((*mask_patch.shape, 4))  # RGBA
                border_overlay[border] = [1, 0, 0, 0.8]  # red border
                ax.imshow(border_overlay)
                # also plot inside of mask but make it transparent
                inside_overlay = np.zeros((*mask_patch.shape, 4))  # RGBA
                inside_overlay[mask_patch] = [1, 0, 0, 0.3]  # transparent red inside
                ax.imshow(inside_overlay)

                if y == 0:
                    ax.set_ylabel(unit_idx, fontsize=20, rotation=0, pad=20)
                    ax.set_yticks([])
                    ax.set_xticks([])
                else:
                    ax.axis("off")
                scalebar = AnchoredSizeBar(
                    ax.transData,
                    # 5 µm is the size of the scalebar
                    8.3,
                    "",
                    "lower right",
                    pad=0.2,
                    color="white",
                    frameon=False,
                    label_top=True,
                    size_vertical=1.5,
                )
                ax.add_artist(scalebar)

            except:
                ax.remove()
# Adjust layout to prevent overlap
plt.tight_layout()
# Show the plot
plt.show()